In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Transpiler passes berkuasa AI

Transpiler passes berkuasa AI adalah laluan yang berfungsi sebagai pengganti langsung laluan Qiskit "tradisional" untuk beberapa tugas transpilasi. Ia sering menghasilkan keputusan yang lebih baik daripada algoritma heuristik sedia ada (seperti kedalaman dan bilangan CNOT yang lebih rendah), tetapi juga jauh lebih pantas daripada algoritma pengoptimuman seperti penyelesai kepuasan Boolean. Transpiler passes AI dijalankan di persekitaran tempatan anda.

> **Note:** Transpiler passes berkuasa AI berada dalam status keluaran beta, tertakluk kepada perubahan.
> Jika anda mempunyai maklum balas atau ingin menghubungi pasukan pembangun, gunakan [saluran Ruang Kerja Qiskit Slack](https://qiskit.slack.com/archives/C06KF8YHUAU) ini.

Laluan berikut tersedia pada masa ini:

**Laluan penghalaan**
 - `AIRouting`: Pemilihan reka letak dan penghalaan Circuit

**Laluan sintesis Circuit**
 - `AICliffordSynthesis`: Sintesis Circuit Clifford
 - `AILinearFunctionSynthesis`: Sintesis Circuit fungsi linear
 - `AIPermutationSynthesis`: Sintesis Circuit permutasi

Untuk menggunakan transpiler passes AI, mula-mula pasang pakej `qiskit-ibm-transpiler`. Lawati [dokumentasi API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) untuk mendapatkan lebih banyak maklumat tentang pilihan yang tersedia.

## Jalankan transpiler passes AI secara tempatan atau di awan
Mula-mula pasang `qiskit-ibm-transpiler` dengan beberapa kebergantungan tambahan seperti berikut:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

backend = QiskitRuntimeService().backend("ibm_fez")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)
transpiled_circuit = ai_passmanager.run(circuit)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Selepas memasang kebergantungan tambahan, mod lalai untuk menjalankan transpiler passes berkuasa AI adalah menggunakan mesin tempatan anda.

## Laluan penghalaan AI
Laluan `AIRouting` bertindak sebagai peringkat reka letak dan peringkat penghalaan. Ia boleh digunakan dalam `PassManager` seperti berikut:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit.circuit.library import efficient_su2

ibm_kingston = QiskitRuntimeService().backend("ibm_kingston")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_kingston,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_kingston, local_mode=True
        ),  # Re-synthesize Linear Function blocks
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Fetching 127 files:   0%|          | 0/127 [00:00<?, ?it/s]

The synthesis respects the coupling map of the device: it can be run safely after other routing passes without disturbing the circuit, so the overall circuit will still follow the device restrictions. By default, the synthesis will replace the original sub-circuit only if the synthesized sub-circuit improves the original (currently only checking CNOT count), but this can be forced to always replace the circuit by setting `replace_only_if_better=False`.

The following synthesis passes are available from `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis for [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) circuits (blocks of `H`, `S`, and `CX` gates). Currently up to nine qubit blocks.
- *AILinearFunctionSynthesis*: Synthesis for [Linear Function](/docs/api/qiskit/qiskit.circuit.library.LinearFunction) circuits (blocks of `CX` and `SWAP` gates). Currently up to nine qubit blocks.
- *AIPermutationSynthesis*: Synthesis for [Permutation](/docs/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuits (blocks of `SWAP` gates). Currently available for 65, 33, and 27 qubit blocks.
- *AIPauliNetworkSynthesis*: Synthesis for Pauli Network circuits (blocks of `H`, `S`, `SX`, `CX`, `RX`, `RY` and `RZ` gates). Currently up to six qubit blocks.

We expect to gradually increase the size of the supported blocks.

All passes use a thread pool to send several requests in parallel. By default, the number for max threads is the number of cores plus four (default values for the `ThreadPoolExecutor` Python object). However, you can set your own value with the `max_threads` argument at pass instantiation. For example, the following line instantiates the `AILinearFunctionSynthesis` pass, which allows it to use a maximum of 20 threads.

```python
AILinearFunctionSynthesis(backend=ibm_torino, max_threads=20)  # Re-synthesize Linear Function blocks using 20 threads max
```

You can also set the environment variable `AI_TRANSPILER_MAX_THREADS` to the desired number of maximum threads, and all synthesis passes instantiated after that will use that value.

For the AI synthesis passes to synthesize a sub-circuit, it must lay on a connected subgraph of the coupling map (one way to do this is with a routing pass before collecting the blocks, but this is not the only way to do it). The synthesis passes will automatically check that the specific subgraph is supported, and if not, it will raise a warning and leave the original sub-circuit unchanged.

The following custom collection passes for Cliffords, Linear Functions and Permutations that can be imported from `qiskit_ibm_transpiler.ai.collection` also complement the synthesis passes:

- *CollectCliffords*: Collects Clifford blocks as `Instruction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectLinearFunctions*: Collects blocks of `SWAP` and `CX` as `LinearFunction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectPermutations*: Collects blocks of `SWAP` circuits as `Permutations`.
- *CollectPauliNetworks*: Collects Pauli Network blocks and stores the original sub-circuit to compare against it after synthesis.

These custom collection passes limit the sizes of the collected sub-circuits so they are supported by the AI-powered synthesis passes. Therefore, it is recommended to use them after the routing passes and before the synthesis passes for a better overall optimization.

## Hybrid heuristic-AI circuit transpilation

The `qiskit-ibm-transpiler` allows you to configure a hybrid pass manager that combines the best of Qiskit's heuristic and the AI-powered transpiler passes. This feature behaves similarly to the Qiskit `generate_pass_manager` method. A typical way to use `generate_ai_pass_manager` is as follows:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_kingston")
kingston_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=kingston_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Di sini, `backend` menentukan peta gandingan yang hendak dihalakan, `optimization_level` (1, 2, atau 3) menentukan usaha pengiraan yang digunakan dalam proses (lebih tinggi biasanya memberikan keputusan lebih baik tetapi mengambil masa lebih lama), dan `layout_mode` menentukan cara mengendalikan pemilihan reka letak.
`layout_mode` termasuk pilihan berikut:

- `keep`: Ini menghormati reka letak yang ditetapkan oleh transpiler passes sebelumnya (atau menggunakan reka letak trivial jika tidak ditetapkan). Ia biasanya hanya digunakan apabila litar mesti dijalankan pada qubit tertentu peranti. Ia sering menghasilkan keputusan yang lebih buruk kerana mempunyai lebih sedikit ruang untuk pengoptimuman.
- `improve`: Ini menggunakan reka letak yang ditetapkan oleh transpiler passes sebelumnya sebagai titik permulaan. Ia berguna apabila anda mempunyai tekaan awal yang baik untuk reka letak; contohnya, untuk litar yang dibina dengan cara yang kira-kira mengikuti peta gandingan peranti. Ia juga berguna jika anda ingin mencuba laluan reka letak tertentu lain yang digabungkan dengan laluan `AIRouting`.
- `optimize`: Ini adalah mod lalai. Ia berfungsi terbaik untuk litar umum di mana anda mungkin tidak mempunyai tekaan reka letak yang baik. Mod ini mengabaikan pemilihan reka letak sebelumnya.
- `local_mode`: Bendera ini menentukan di mana laluan `AIRouting` dijalankan. Jika `False`, `AIRouting` dijalankan dari jauh melalui Perkhidmatan Transpiler Qiskit. Jika `True`, pakej cuba menjalankan laluan di persekitaran tempatan anda dengan sandaran ke mod awan jika kebergantungan yang diperlukan tidak ditemui.

## Laluan sintesis Circuit AI
Laluan sintesis Circuit AI membolehkan anda mengoptimumkan bahagian jenis litar yang berbeza ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Fungsi Linear](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutasi](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Rangkaian Pauli) dengan mensintesiskan semula. Cara biasa untuk menggunakan laluan sintesis adalah seperti berikut: